In [18]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import sys

# Change root directory to the repo root (Jupyter: __file__ is not defined)
def find_repo_root(start=Path.cwd()):
	for p in [start] + list(start.parents):
		if (p / 'Code').exists() or (p / '.git').exists() or (p / 'local_repo').exists():
			return p
	return start

repo_root = find_repo_root()
data_root = repo_root.parent.parent / 'Data' / 'CBOS ready'

os.chdir(repo_root)
sys.path.append(str(repo_root / 'Code' / 'tools'))

JSON_ROOT = "/Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/CT json"

In [19]:
import ast

df = pd.read_csv(data_root / 'CBOS_data_to_weight.csv')
print(f"Loaded: {df.shape[0]:,} obs x {df.shape[1]} cols, {df['survey_file'].nunique()} surveys")

# Parse teryt_id_VOIV: strings for voivodeships, stringified lists for macroregions
def parse_voiv(x):
    s = str(x)
    if s.startswith('['):
        return ast.literal_eval(s)
    return s.zfill(7) if s != 'nan' and s != 'None' else None

df['teryt_id_VOIV'] = df['teryt_id_VOIV'].apply(parse_voiv)

# Parse teryt_id list columns
for col in ['teryt_id_VOIV_500', 'teryt_id_VOIV_100_500', 'teryt_id_VOIV_100']:
    df[col] = df[col].apply(lambda x: ast.literal_eval(str(x)) if pd.notna(x) and str(x).startswith('[') else [])

# Parse survey_date
df['survey_date'] = pd.to_datetime(df['survey_date'])

print(f"teryt_id_VOIV types: str={sum(isinstance(v, str) for v in df['teryt_id_VOIV'])}, "
      f"list={sum(isinstance(v, list) for v in df['teryt_id_VOIV'])}, "
      f"none={sum(v is None for v in df['teryt_id_VOIV'])}")

/var/folders/y8/4_9g68pj7k136q2yypgp5ysc0000gn/T/ipykernel_15530/2524629453.py:3: DtypeWarning: Columns (15,16,21,27,28,29,33,34,35,40,41,45,46,47,79) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_root / 'CBOS_data_to_weight.csv')


Loaded: 355,337 obs x 88 cols, 327 surveys
teryt_id_VOIV types: str=345328, list=10009, none=0


In [20]:
# --- Diagnostic: Identify problematic cs values ---
print("cs_prob distribution:")
print(df['cs_prob'].value_counts(dropna=False).sort_index())
print()

prob = df[df['cs_prob'].isna()].copy()
print(f"Problematic cases: {len(prob)}")
print(f"  cs=4.0: {(prob['cs']==4.0).sum()}")
print(f"  cs=5.0: {(prob['cs']==5.0).sum()}")
print()
print("By period:")
print(f"  Pre-1999 (old voiv): {len(prob[prob['survey_year'] < 1999])}")
print(f"  1999-2007 (both):    {len(prob[(prob['survey_year'] >= 1999) & (prob['survey_year'] <= 2007)])}")
print(f"  2008+ (new voiv):    {len(prob[prob['survey_year'] >= 2008])}")

cs_prob distribution:
cs_prob
1.0    136091
2.0     45677
3.0     69183
4.0     63224
5.0     38735
NaN      2427
Name: count, dtype: int64

Problematic cases: 2427
  cs=4.0: 717
  cs=5.0: 1710

By period:
  Pre-1999 (old voiv): 2261
  1999-2007 (both):    146
  2008+ (new voiv):    20


In [21]:
# --- Cross-tab analysis of problematic cases ---
prob = df[df['cs_prob'].isna()].copy()

print("=== Problematic cases: cs x location_old_L ===")
ct_old = prob[prob['location_old'].notna()].groupby(['location_old_L', 'cs']).size().unstack(fill_value=0)
print(ct_old.to_string())
print()
print("=== Problematic cases: cs x location_new_L ===")
ct_new = prob[prob['location_new'].notna()].groupby(['location_new_L', 'cs']).size().unstack(fill_value=0)
print(ct_new.to_string())

=== Problematic cases: cs x location_old_L ===
cs                4.0  5.0
location_old_L            
bialskopodlaskie   96   42
białostockie        0   61
bielskie            0   67
bydgoskie           0  204
chełmskie          12    0
ciechanowskie       0    1
częstochowskie      1   14
elbląskie           0    1
gdańskie            0  479
jeleniogórskie     78    0
kaliskie            0    3
katowickie          0  539
kieleckie           0   17
konińskie           8    0
krakowskie         24    0
krośnieńskie        1    3
legnickie           0    1
lubelskie           0   11
nowosądeckie        1    1
olsztyńskie         0    5
opolskie            0    6
ostrołęckie         2    0
pilskie             6    0
piotrkowskie       13    2
poznańskie         34    0
przemyskie          1    0
płockie             0    2
radomskie           0    8
rzeszowskie         0    1
siedleckie          5    0
skierniewickie      8    3
suwalskie           5    3
szczecińskie        0   49
tarnobrz

In [22]:
# --- For each problematic voivodeship, find max working cs ---
print("=== Old voivodeships: max working cs ===")
for voiv_name in sorted(prob[prob['location_old'].notna()]['location_old_L'].unique()):
    ok = df[(df['location_old_L'] == voiv_name) & (df['cs_prob'].notna())]
    prob_cs = prob[prob['location_old_L'] == voiv_name]['cs'].unique()
    max_cs = ok['cs'].max() if len(ok) > 0 else 'NONE'
    print(f"  {voiv_name}: max_ok={max_cs}, problematic={sorted(prob_cs)}")

print("\n=== New voivodeships: max working cs ===")
for voiv_name in sorted(prob[prob['location_new'].notna()]['location_new_L'].unique()):
    ok = df[(df['location_new_L'] == voiv_name) & (df['cs_prob'].notna())]
    prob_cs = prob[prob['location_new_L'] == voiv_name]['cs'].unique()
    max_cs = ok['cs'].max() if len(ok) > 0 else 'NONE'
    print(f"  {voiv_name}: max_ok={max_cs}, problematic={sorted(prob_cs)}")

=== Old voivodeships: max working cs ===
  bialskopodlaskie: max_ok=3.0, problematic=[np.float64(4.0), np.float64(5.0)]
  białostockie: max_ok=4.0, problematic=[np.float64(5.0)]
  bielskie: max_ok=4.0, problematic=[np.float64(5.0)]
  bydgoskie: max_ok=4.0, problematic=[np.float64(5.0)]
  chełmskie: max_ok=3.0, problematic=[np.float64(4.0)]
  ciechanowskie: max_ok=3.0, problematic=[np.float64(5.0)]
  częstochowskie: max_ok=4.0, problematic=[np.float64(4.0), np.float64(5.0)]
  elbląskie: max_ok=4.0, problematic=[np.float64(5.0)]
  gdańskie: max_ok=4.0, problematic=[np.float64(5.0)]
  jeleniogórskie: max_ok=3.0, problematic=[np.float64(4.0)]
  kaliskie: max_ok=4.0, problematic=[np.float64(5.0)]
  katowickie: max_ok=4.0, problematic=[np.float64(5.0)]
  kieleckie: max_ok=4.0, problematic=[np.float64(5.0)]
  konińskie: max_ok=3.0, problematic=[np.float64(4.0)]
  krakowskie: max_ok=5.0, problematic=[np.float64(4.0)]
  krośnieńskie: max_ok=3.0, problematic=[np.float64(4.0), np.float64(5.0)]
  

In [23]:
# --- Pattern summary ---
print("=== Deterministic correction patterns ===")
cs5 = prob[prob['cs'] == 5.0]
cs4 = prob[prob['cs'] == 4.0]

# cs=5.0: voivodeship has no 500k+ city -> downgrade
# Check if 100-500k cities exist (cs=4.0), else 20-100k (cs=3.0)
print(f"Pattern A: cs=5.0 -> down: {len(cs5)} cases")

# cs=4.0: two subcases
# Subcase 1: voiv has 500k+ but no 100-500k -> upgrade to 5.0
# Subcase 2: voiv has no city >= 100k -> downgrade to 3.0
upgrade_voivs_old = ['krakowskie', 'wrocławskie', 'warszawskie', 'łódzkie', 'poznańskie']
p4_upgrade = cs4[cs4['location_old_L'].isin(upgrade_voivs_old)]
p4_downgrade = cs4[~cs4.index.isin(p4_upgrade.index)]
print(f"Pattern B-up: cs=4.0 -> 5.0 (has 500k+ city): {len(p4_upgrade)} cases")
print(f"Pattern C-down: cs=4.0 -> 3.0 (no 100k+ city): {len(p4_downgrade)} cases")
print(f"Total: {len(prob)} cases")

=== Deterministic correction patterns ===
Pattern A: cs=5.0 -> down: 1710 cases
Pattern B-up: cs=4.0 -> 5.0 (has 500k+ city): 456 cases
Pattern C-down: cs=4.0 -> 3.0 (no 100k+ city): 261 cases
Total: 2427 cases


In [24]:
# placeholder - kept for notebook structure
pass

In [25]:
# placeholder - kept for notebook structure
pass

In [26]:
# placeholder - kept for notebook structure
pass

In [27]:
# placeholder - kept for notebook structure
pass

In [28]:
# ============================================================================
# DETERMINISTIC CITY SIZE CORRECTION
# ============================================================================
# For each problematic row, find the nearest valid cs category by checking
# which city size categories are valid (produce non-empty teryt_id lists)
# for that voivodeship/year combination.

def voiv_matches(v1, v2):
    """Check if two teryt_id_VOIV values match (handles lists and strings)."""
    if isinstance(v1, list) and isinstance(v2, list):
        return v1 == v2
    if isinstance(v1, str) and isinstance(v2, str):
        return v1 == v2
    return False

def find_corrected_cs(row, df_full):
    """Find the nearest valid cs for a problematic row."""
    original_cs = row['cs']
    voiv = row['teryt_id_VOIV']
    
    # Find rows in the same voivodeship
    same_voiv = df_full[df_full['teryt_id_VOIV'].apply(lambda x: voiv_matches(x, voiv))]
    
    # Find valid cs values (non-NaN cs_prob)
    valid_entries = same_voiv[same_voiv['cs_prob'].notna()]
    valid_cs_values = sorted(valid_entries['cs'].unique())
    
    if len(valid_cs_values) == 0:
        return np.nan
    
    if original_cs == 5.0:
        # Voivodeship has no 500k+ city -> downgrade to highest valid
        candidates = [c for c in valid_cs_values if c < original_cs]
        return max(candidates) if candidates else np.nan
    
    elif original_cs == 4.0:
        # Check: has 500k+ city but no 100-500k? -> upgrade to 5.0
        # No city >= 100k? -> downgrade to 3.0
        if 5.0 in valid_cs_values and 4.0 not in valid_cs_values:
            return 5.0
        elif 4.0 not in valid_cs_values:
            candidates = [c for c in valid_cs_values if c < original_cs]
            return max(candidates) if candidates else np.nan
        else:
            return np.nan
    
    return np.nan

# Apply corrections
prob_mask = df['cs_prob'].isna()
print(f"Correcting {prob_mask.sum()} problematic cs values...")

corrections = {}
for idx in df[prob_mask].index:
    corrections[idx] = find_corrected_cs(df.loc[idx], df)

for idx, corrected_cs in corrections.items():
    df.at[idx, 'cs_prob'] = corrected_cs

# Summary
print(f"Remaining NaN: {df['cs_prob'].isna().sum()}")
correction_df = pd.DataFrame({
    'original_cs': df.loc[list(corrections.keys()), 'cs'],
    'corrected_cs': pd.Series(corrections)
})
print("\nCorrection summary:")
print(correction_df.groupby(['original_cs', 'corrected_cs']).size().reset_index(name='count'))

Correcting 2427 problematic cs values...
Remaining NaN: 0

Correction summary:
   original_cs  corrected_cs  count
0          4.0           3.0    255
1          4.0           5.0    462
2          5.0           3.0     60
3          5.0           4.0   1650


In [29]:
# ============================================================================
# VALIDATION OF CORRECTIONS
# ============================================================================
corrected_rows = df.loc[list(corrections.keys())]

# Check 1: corrected != original
assert (corrected_rows['cs'] != corrected_rows['cs_prob']).all()
print("Check 1 PASS: All corrected cs_prob != original cs")

# Check 2: values in valid range
assert corrected_rows['cs_prob'].between(1.0, 5.0).all()
print("Check 2 PASS: All corrected values in [1.0, 5.0]")

# Check 3: no remaining NaN
remaining = df['cs_prob'].isna().sum()
print(f"Check 3: Remaining NaN = {remaining}")

# Check 4: corrected cs matches vs non-corrected
corrected_mask = df.index.isin(corrections.keys())
non_corrected = ~corrected_mask & df['cs'].notna()
assert (df.loc[non_corrected, 'cs'] == df.loc[non_corrected, 'cs_prob']).all()
print(f"Check 4 PASS: {non_corrected.sum()} non-corrected rows have cs_prob == cs")

# Distribution comparison
print(f"\n{'Value':<6} {'cs':>10} {'cs_prob':>10} {'diff':>8}")
for val in [1.0, 2.0, 3.0, 4.0, 5.0]:
    o = (df['cs'] == val).sum()
    c = (df['cs_prob'] == val).sum()
    print(f"{val:<6} {o:>10} {c:>10} {c-o:>+8}")

print(f"\nCorrection breakdown:")
for (ocs, ncs), cnt in correction_df.groupby(['original_cs','corrected_cs']).size().items():
    print(f"  cs={ocs} -> cs_prob={ncs}: {cnt} cases")

Check 1 PASS: All corrected cs_prob != original cs
Check 2 PASS: All corrected values in [1.0, 5.0]
Check 3: Remaining NaN = 0
Check 4 PASS: 352910 non-corrected rows have cs_prob == cs

Value          cs    cs_prob     diff
1.0        136091     136091       +0
2.0         45677      45677       +0
3.0         69183      69498     +315
4.0         63941      64874     +933
5.0         40445      39197    -1248

Correction breakdown:
  cs=4.0 -> cs_prob=3.0: 255 cases
  cs=4.0 -> cs_prob=5.0: 462 cases
  cs=5.0 -> cs_prob=3.0: 60 cases
  cs=5.0 -> cs_prob=4.0: 1650 cases


In [30]:
# ============================================================================
# LOAD U_ DICTS AND BUILD LOOKUPS (new key format)
# ============================================================================
# Key format: (int(voiv[:2]), int(year), translate_type(type), cs)
# translate_type: '500'->5, '100-200-500'->125, '100-500'->15, '100'->1

import json

out_root = repo_root.parent.parent / 'Data' / 'reweighing'

with open(out_root / 'U_old_voiv_dict.json', 'r') as f:
    U_old = json.load(f)
with open(out_root / 'U_new_voiv_dict.json', 'r') as f:
    U_new = json.load(f)
with open(out_root / 'U_cbos_macroregions_dict.json', 'r') as f:
    U_cbos = json.load(f)
with open(out_root / 'U_lis_macroregions_dict.json', 'r') as f:
    U_lis = json.load(f)

def build_lookup(U_dict):
    """Build flat lookup: (voiv_prefix, year, type_code, cs) -> teryt_id_list
    Also build reverse: same key -> group_id"""
    lookup = {}
    group_lookup = {}
    for group_id, group in U_dict.items():
        array = group['array']
        for key in group['keys']:
            # key = [int_voiv, int_year, int_type, float_cs]
            k = tuple(key)
            lookup[k] = array
            group_lookup[k] = group_id
    return lookup, group_lookup

lookup_old, group_old = build_lookup(U_old)
lookup_new, group_new = build_lookup(U_new)
lookup_cbos, group_cbos = build_lookup(U_cbos)
lookup_lis, group_lis = build_lookup(U_lis)

print(f"Lookup sizes: old={len(lookup_old)}, new={len(lookup_new)}, cbos={len(lookup_cbos)}, lis={len(lookup_lis)}")

# Macroregion definitions
cbos_macroregions = {1: ['1400000', '1000000'],
                     2: ['1200000', '2400000'],
                     3: ['0600000', '1800000', '2000000', '2600000'],
                     4: ['3000000', '3200000', '0800000'],
                     5: ['0200000', '1600000'],
                     6: ['2200000', '2800000', '0400000']}

# translate_type for key construction
def translate_type(t):
    return {'500': 5, '100-200-500': 125, '100-500': 15, '100': 1}.get(t)

def make_key(voiv, year, type_str, cs):
    """Create lookup key from voiv code, year, type string, and cs."""
    if isinstance(voiv, list):
        # Macroregion: find which macroregion number
        for mr_num, mr_voivs in cbos_macroregions.items():
            if voiv == mr_voivs:
                return (mr_num, int(year), translate_type(type_str), cs)
        return None
    else:
        return (int(str(voiv)[:2]), int(year), translate_type(type_str), cs)

def lookup_teryt(voiv, year, type_str, cs):
    """Look up teryt_ids from the appropriate dictionary."""
    key = make_key(voiv, year, type_str, cs)
    if key is None:
        return None
    if isinstance(voiv, list):
        return lookup_cbos.get(key)
    result = lookup_new.get(key)
    if result is None:
        result = lookup_old.get(key)
    return result

def lookup_group(voiv, year, type_str, cs):
    """Look up group ID from the appropriate dictionary."""
    key = make_key(voiv, year, type_str, cs)
    if key is None:
        return None
    if isinstance(voiv, list):
        return group_cbos.get(key)
    result = group_new.get(key)
    if result is None:
        result = group_old.get(key)
    return result

# Test
test = lookup_teryt('1400000', 2010, '100-500', 4.0)
print(f"Test ('1400000', 2010, '100-500', 4.0): {test is not None}, len={len(test) if test else 0}")
test_g = lookup_group('1400000', 2010, '100-500', 4.0)
print(f"Test group: {test_g}")

Lookup sizes: old=40131, new=13104, cbos=4914, lis=7371
Test ('1400000', 2010, '100-500', 4.0): True, len=2
Test group: Group 118


In [31]:
# ============================================================================
# UPDATE TERYT_ID COLUMNS FOR CORRECTED ROWS
# ============================================================================
corrected_indices = list(corrections.keys())
updates = {'500': 0, '100_500': 0, '100': 0}
failed = 0

for idx in corrected_indices:
    row = df.loc[idx]
    voiv = row['teryt_id_VOIV']
    cs = float(row['cs_prob'])
    year = int(pd.Timestamp(row['survey_date']).year)
    
    for type_str, col_suffix in [('500', '500'), ('100-500', '100_500'), ('100', '100')]:
        col = f'teryt_id_VOIV_{col_suffix}'
        result = lookup_teryt(voiv, year, type_str, cs)
        if result is not None:
            df.at[idx, col] = result
            updates[col_suffix] += 1
        else:
            failed += 1

print(f"Updated teryt_id columns for {len(corrected_indices)} corrected rows:")
for k, v in updates.items():
    print(f"  teryt_id_VOIV_{k}: {v} updated")
print(f"  Failed lookups: {failed}")

# Verify no empty lists remain for corrected rows
for col_suffix in ['500', '100_500', '100']:
    col = f'teryt_id_VOIV_{col_suffix}'
    empty = df.loc[corrected_indices, col].apply(
        lambda x: len(x) == 0 if isinstance(x, list) else True).sum()
    print(f"  Empty {col}: {empty}")

Updated teryt_id columns for 2427 corrected rows:
  teryt_id_VOIV_500: 2427 updated
  teryt_id_VOIV_100_500: 2427 updated
  teryt_id_VOIV_100: 2427 updated
  Failed lookups: 0
  Empty teryt_id_VOIV_500: 0
  Empty teryt_id_VOIV_100_500: 0
  Empty teryt_id_VOIV_100: 0


In [32]:
# ============================================================================
# CREATE G_VOIV GROUP VARIABLES
# ============================================================================
# For each observation, assign its U_ dict group ID based on corrected cs_prob.
# This avoids re-comparing teryt_id arrays later.

print("Creating G_VOIV group variables...")

for type_str, col_suffix in [('500', '500'), ('100-500', '100_500'), ('100', '100')]:
    col = f'G_VOIV_{col_suffix}'
    groups = []
    for idx, row in df.iterrows():
        voiv = row['teryt_id_VOIV']
        cs = row['cs_prob']
        if pd.isna(cs) or voiv is None:
            groups.append(None)
            continue
        year = int(pd.Timestamp(row['survey_date']).year)
        g = lookup_group(voiv, year, type_str, cs)
        groups.append(g)
    df[col] = groups
    n_groups = df[col].nunique()
    n_none = df[col].isna().sum()
    print(f"  {col}: {n_groups} unique groups, {n_none} missing")

print("Done.")

Creating G_VOIV group variables...
  G_VOIV_500: 120 unique groups, 0 missing
  G_VOIV_100_500: 151 unique groups, 0 missing
  G_VOIV_100: 146 unique groups, 0 missing
Done.


In [33]:
# ============================================================================
# FINAL VERIFICATION
# ============================================================================

# 1. Verify corrected rows have non-empty teryt lists
def is_empty_teryt(x):
    if isinstance(x, list):
        return len(x) == 0
    return True

for col_suffix in ['500', '100_500', '100']:
    col = f'teryt_id_VOIV_{col_suffix}'
    empty_corr = df.loc[corrected_indices, col].apply(is_empty_teryt).sum()
    print(f"Corrected rows with empty {col}: {empty_corr}/{len(corrected_indices)}")

# 2. Show sample corrected values
print(f"\nSample corrected rows:")
for idx in corrected_indices[:5]:
    row = df.loc[idx]
    print(f"  idx={idx}: cs {row['cs']}→{row['cs_prob']}, "
          f"voiv={row['teryt_id_VOIV']}, "
          f"G_500={row.get('G_VOIV_500')}, G_100_500={row.get('G_VOIV_100_500')}, G_100={row.get('G_VOIV_100')}")

# 3. Verify G_VOIV columns
for col in ['G_VOIV_500', 'G_VOIV_100_500', 'G_VOIV_100']:
    if col in df.columns:
        n_valid = df[col].notna().sum()
        n_unique = df[col].nunique()
        print(f"\n{col}: {n_valid}/{len(df)} valid, {n_unique} unique groups")

# 4. Verify no NaN in cs_prob
remaining_nan = df['cs_prob'].isna().sum()
print(f"\nRemaining cs_prob NaN: {remaining_nan}")

# 5. Overall dataset summary
print(f"\nFinal dataset: {df.shape[0]:,} rows x {df.shape[1]} cols")
print(f"Columns: {list(df.columns)}")

Corrected rows with empty teryt_id_VOIV_500: 0/2427
Corrected rows with empty teryt_id_VOIV_100_500: 0/2427
Corrected rows with empty teryt_id_VOIV_100: 0/2427

Sample corrected rows:
  idx=2: cs 5.0→4.0, voiv=9600000, G_500=Group 9, G_100_500=Group 11, G_100=Group 11
  idx=3: cs 5.0→4.0, voiv=9600000, G_500=Group 9, G_100_500=Group 11, G_100=Group 11
  idx=4: cs 5.0→4.0, voiv=9800000, G_500=Group 3, G_100_500=Group 5, G_100=Group 5
  idx=5: cs 5.0→3.0, voiv=9900000, G_500=Group 1, G_100_500=Group 1, G_100=Group 1
  idx=6: cs 5.0→3.0, voiv=9900000, G_500=Group 1, G_100_500=Group 1, G_100=Group 1

G_VOIV_500: 355337/355337 valid, 120 unique groups

G_VOIV_100_500: 355337/355337 valid, 151 unique groups

G_VOIV_100: 355337/355337 valid, 146 unique groups

Remaining cs_prob NaN: 0

Final dataset: 355,337 rows x 91 cols
Columns: ['Unnamed: 0', 'org_id', 'survey_file', 'survey_year', 'survey_month', 'age', 'year_born', 'sex', 'sex_L', 'city_size', 'city_size_L', 'education', 'education_L', 

In [34]:
# Save the corrected DataFrame to a new CSV
output_path = data_root / 'CBOS_data_to_weight_corrected.csv'
df.to_csv(output_path, index=False)
print(f"\nCorrected data saved to: {output_path}")


Corrected data saved to: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/CBOS ready/CBOS_data_to_weight_corrected.csv
